# Эксперименты с нейросетями: анализ тональности текста

Вы работаете в команде картографического сервиса YouMap, который собирает отзывы о компаниях, организациях, квартирах и других заведениях из разных источников. Команде качества нужно научиться автоматически находить негативные сообщения и передавать их на приоритетную обработку. Ваша задача — подготовить данные, обучить модель и объяснить, какой подход лучше помогает находить негативные отзывы.

Вам предстоит использовать датасет `ds_s26_geo_reviews.csv`.

|Признак|Описание|Комментарий|
|:---|:---|:---|
|text|Текст отзыва|Отзывы для обучения нейросети|
|address|Адрес организации|Только для EDA|
|federal_district|Федеральный округ|Категориальный признак|
|name_ru|Название организации|Только для EDA. <br> Не используйте его для обучения|
|rubrics|Категории, к которым относится организация|Только для EDA|
|rubrics_count|Количество категорий, к которым относится организация|Числовой признак|
|review_len_chars|Длина отзыва в символах|Числовой признак|
|review_len_words|Длина отзыва в словах|Числовой признак|
|emoji_count|Количество эмодзи в отзыве|Числовой признак|
|target|1 — негативный отзыв, оценка 1–2. <br> 0 — остальные отзывы, оценка 3–5.|Целевая переменная|

В проекте вы сравните два способа представить один и тот же текст для нейросети:

*   Числовые признаки текста + MLP.
*   Индексы токенов + эмбеддинги + mean pool +  MLP.

Основная метрика — PR-AUC, вспомогательные — ROC-AUC, precision, recall, F1-мера.

В ячейках с пропусками (`...`) допишите недостающий код. Не меняйте названия ключевых переменных: они используются в следующих блоках.

## Шаг 1. Импорт библиотек, настройка констант, обеспечение воспроизводимости

In [ ]:
from pathlib import Path
from collections import Counter
import random
import re
import string

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

from IPython.display import display

In [ ]:
RANDOM_STATE = 42
DATA_PATH = Path("/datasets/ds_s26_geo_reviews")
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)


def set_seed(seed: int = RANDOM_STATE) -> None:
    # Зафиксируйте random seed для random, numpy и torch
    random.seed(...)
    np.random.seed(...)
    torch.manual_seed(...)
    torch.cuda.manual_seed_all(...)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(RANDOM_STATE)

# Определите устройство: cuda по доступности, иначе cpu
device = torch.device(...)
device

## Шаг 2. Загрузка данных и исследовательский анализ

EDA в этом проекте — короткий блок для знакомства с данными, в котором весь код уже написан. Он поможет понять размер датасета, проанализировать целевую переменную, проверить дисбаланс классов и пропуски.

Запустите ячейки, посмотрите на результаты и переходите к оформлению выводов.

In [ ]:
# Код для EDA готов: запустите ячейки и изучите выводы
df = pd.read_csv(DATA_PATH)

print(f"Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов")
display(df.head())

In [ ]:
# Типы данных, пропуски и количество уникальных значений
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique(),
}))

In [ ]:
# Распределение классов: количество объектов каждого класса и их доля
target_distribution = (
    df["target"]
    .value_counts()
    .sort_index()
    .rename_axis("target")
    .reset_index(name="count")
)
target_distribution["share"] = target_distribution["count"] / len(df)

display(target_distribution)

plt.figure(figsize=(5, 3))
plt.bar(target_distribution["target"].astype(str), target_distribution["count"])
plt.title("Распределение классов")
plt.xlabel("target")
plt.ylabel("Количество отзывов")
plt.show()

In [ ]:
pd.set_option("display.max_colwidth", 220)

# Примеры объектов каждого класса: посмотрите, как выглядит текстовая разметка
for class_value, title in [(0, "ненегативные отзывы"), (1, "негативные отзывы")]:
    print(f"Примеры: {title}")
    examples = (
        df[df["target"] == class_value]
        [["text", "federal_district", "rubrics_count", "review_len_words", "emoji_count", "target"]]
        .sample(3, random_state=RANDOM_STATE)
    )
    display(examples)

Сформулируйте выводы, кратко ответив на вопросы:

* Насколько классы сбалансированы?

* Почему accuracy здесь может вводить в заблуждение?

* Какие пропуски обнаружились и как с ними нужно поступить?

* Почему важно помнить, что `target` получен из пользовательских оценок, а не ручной разметкой текста?

In [ ]:
# Напишите здесь решение для устранения пропусков, если обнаружили их

## Шаг 3. Подготовка числовых признаков

Для первого эксперимента используйте готовые числовые признаки, закодированный признак `federal_district` и самостоятельно извлечённые из текста числовые показатели:

- `punct_count` — количество знаков препинания.
- `question_count` — количество вопросительных знаков.
- `exclamation_count` — количество восклицательных знаков.
- `newline_count` — количество переносов строк.
- `digit_count` — количество цифр.
- `uppercase_share` — доля заглавных букв среди всех букв.
- `avg_word_len` — средняя длина слова.

Выделив все признаки, разделите данные на обучающую, валидационную и тестовую выборки, а затем подготовьте данные для нейросети, создав объекты классов `Dataset` и `DataLoader`. Это же разделение нужно будет использовать и в эксперименте с текстовой моделью, чтобы сравнение было честным.

Не используйте признаки `name_ru`, `address` и `rubrics` в модели: они нужны для знакомства с данными, но могут привести к запоминанию конкретных организаций или слишком детальных категорий.

In [ ]:
df_features = df.copy()
text = df_features["text"].astype(str)
punctuation = string.punctuation + "«»—–…“”„"

# 1. Количество знаков препинания
df_features["punct_count"] = ...

# 2. Количество вопросительных знаков
df_features["question_count"] = ...

# 3. Количество восклицательных знаков
df_features["exclamation_count"] = ...

# 4. Количество переносов строк
df_features["newline_count"] = ...

# 5. Количество цифр
df_features["digit_count"] = ...

# 6. Доля заглавных букв среди всех букв
# Сначала посчитайте общее число букв и число заглавных букв
letters_count = ...
upper_count = ...
df_features["uppercase_share"] = ...
df_features["uppercase_share"] = df_features["uppercase_share"].fillna(0)

# 7. Средняя длина слова
df_features["avg_word_len"] = ...
df_features["avg_word_len"] = df_features["avg_word_len"].fillna(0)

numeric_features = [
    "rubrics_count",
    "emoji_count",
    "punct_count",
    "question_count",
    "exclamation_count",
    "newline_count",
    "digit_count",
    "uppercase_share",
    "avg_word_len",
]

categorical_features = ["federal_district"]

display(df_features[numeric_features + ["target"]].describe().T)

district_distribution = (
    df_features["federal_district"]
    .value_counts()
    .rename_axis("federal_district")
    .reset_index(name="count")
)
district_distribution["share"] = district_distribution["count"] / len(df_features)
display(district_distribution)

In [ ]:
# Разделите данные на обучающую, валидационную и тестовую выборки в пропорции 70:15:15
# Не забудьте про стратификацию, чтобы сохранить долю негативных отзывов
train_df, temp_df = train_test_split(
    df_features,
    test_size=...,
    stratify=...,
    random_state=RANDOM_STATE,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=...,
    stratify=...,
    random_state=RANDOM_STATE,
)

split_stats = pd.DataFrame(
    {
        "rows": [len(train_df), len(val_df), len(test_df)],
        "target_share": [
            train_df["target"].mean(),
            val_df["target"].mean(),
            test_df["target"].mean(),
        ],
    },
    index=["обучающая", "валидационная", "тестовая"],
)

display(split_stats)

In [ ]:
# Создайте scaler для числовых признаков и encoder для federal_district
scaler = StandardScaler()
one_hot_encoder = OneHotEncoder(...)

# Важно: fit только на обучающей выборке, для валидационной и тестовой используйте transform
X_train_num = scaler.fit_transform(...)
X_val_num = scaler.transform(...)
X_test_num = scaler.transform(...)

X_train_cat = one_hot_encoder.fit_transform(...)
X_val_cat = one_hot_encoder.transform(...)
X_test_cat = one_hot_encoder.transform(...)

X_train = np.hstack([X_train_num, X_train_cat]).astype("float32")
X_val = np.hstack([X_val_num, X_val_cat]).astype("float32")
X_test = np.hstack([X_test_num, X_test_cat]).astype("float32")

y_train = train_df["target"].values.astype("float32")
y_val = val_df["target"].values.astype("float32")
y_test = test_df["target"].values.astype("float32")

feature_names = numeric_features + list(
    one_hot_encoder.get_feature_names_out(categorical_features)
)

print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)
print("Количество признаков после кодирования:", len(feature_names))
display(pd.DataFrame({"feature": feature_names}))

In [ ]:
# Создание Dataset и DataLoader

class ReviewsDataset(Dataset):
    def __init__(self, features: np.ndarray, target: np.ndarray):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.target = torch.tensor(target, dtype=torch.float32)

    def __len__(self):
        # Верните количество объектов в датасете
        return ...

    def __getitem__(self, idx):
        # Верните пару: признаки объекта и целевую метку
        return ...


BATCH_SIZE = 128

numeric_loaders = {
    "train": DataLoader(
        ReviewsDataset(...),
        batch_size=BATCH_SIZE,
        shuffle=True,
    ),
    "val": DataLoader(
        ReviewsDataset(...),
        batch_size=BATCH_SIZE,
        shuffle=False,
    ),
    "test": DataLoader(
        ReviewsDataset(...),
        batch_size=BATCH_SIZE,
        shuffle=False,
    ),
}

features_batch, target_batch = next(iter(numeric_loaders["train"]))
print("features_batch:", features_batch.shape, features_batch.dtype)
print("target_batch:", target_batch.shape, target_batch.dtype)

## Шаг 4. Эксперимент 1 — MLP на числовых признаках

Обучите несколько MLP-конфигураций. Во всех экспериментах используйте одно и то же разделение на выборки и одну и ту же функцию потерь.

Дополните код функций:

- `train_epoch` — нужна для одного прохода по обучающей выборке.
- `evaluate` — нужна для оценки на валидационной или тестовой выборках.

Функция `run_experiment` уже готова, её можно переиспользовать.

Из-за дисбаланса классов нужно добавить коэффициент `pos_weight` в `BCEWithLogitsLoss`. Это созданный для работы с нейросетями вариант модификации (взвешивания) функции потерь — знакомого вам метода борьбы с дисбалансом классов. Благодаря `pos_weight` ошибки на редком классе 1 получат больший вес. Это не отменяет необходимости выбирать конфигурации по PR-AUC и дополнительно анализировать ROC-AUC, precision, recall и F1.

In [ ]:
def calc_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> dict:
    # PR-AUC и ROC-AUC считайте по вероятностям y_prob
    # Для precision, recall, F1 и accuracy сначала переведите вероятности в классы
    # В этом проекте используйте порог по умолчанию: threshold = 0.5
    y_pred = ...

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )

    return {
        "pr_auc": ...,
        "roc_auc": ...,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": ...,
    }


majority_accuracy = (test_df["target"] == 0).mean()
print(f"Accuracy константной модели, которая всегда предсказывает 0: {majority_accuracy:.3f}")

In [ ]:
# Класс модели и обучение
class MLP(nn.Module):
    def __init__(self, input_dim: int, hidden_layers=(32, 16), dropout: float = 0.0):
        super().__init__()
        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_layers:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # Опишите прямой проход модели
        return ...

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for features, target in loader:
        features = features.to(device)
        target = target.to(device)

        # Выполните шаг обучения:
        # zero_grad → forward → loss → backward → step
        optimizer.zero_grad()
        logits = ...
        loss = ...
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(target)

    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_probs = []
    all_targets = []

    with torch.no_grad():
        for features, target in loader:
            features = features.to(device)
            target = target.to(device)

            # Выполните forward pass и посчитайте loss
            logits = ...
            loss = ...
            probs = ...

            total_loss += loss.item() * len(target)
            all_probs.extend(probs.cpu().numpy())
            all_targets.extend(target.cpu().numpy())

    y_true = np.array(all_targets)
    y_prob = np.array(all_probs)
    metrics = calc_metrics(y_true, y_prob)
    metrics["loss"] = total_loss / len(loader.dataset)

    return metrics, y_true, y_prob

In [ ]:
pos_weight_value = (y_train == 0).sum() / (y_train == 1).sum()
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f"pos_weight: {pos_weight_value:.2f}")


def run_experiment(model, loaders: dict, config: dict):
    model = model.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    history = []
    best_state = None
    best_val_pr_auc = -np.inf

    for epoch in range(1, config["epochs"] + 1):
        train_loss = train_epoch(model, loaders["train"], criterion, optimizer, device)
        val_metrics, _, _ = evaluate(model, loaders["val"], criterion, device)

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                **val_metrics,
            }
        )

        if val_metrics["pr_auc"] > best_val_pr_auc:
            best_val_pr_auc = val_metrics["pr_auc"]
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

In [ ]:
mlp_configs = [
    {
        "name": "small_no_reg",
        "hidden_layers": (16,),
        "dropout": 0.0,
        "lr": 0.001,
        "weight_decay": 0.0,
        "epochs": 8,
    },
    {
        "name": "deep_no_reg",
        "hidden_layers": (64, 32),
        "dropout": 0.0,
        "lr": 0.001,
        "weight_decay": 0.0,
        "epochs": 8,
    },
    {
        "name": "regularized",
        "hidden_layers": (64, 32),
        "dropout": 0.3,
        "lr": 0.001,
        "weight_decay": 0.0001,
        "epochs": 8,
    },
]

In [ ]:
mlp_histories = {}
mlp_models = {}
mlp_rows = []

for config in mlp_configs:
    set_seed(RANDOM_STATE)

    model = MLP(
        input_dim=X_train.shape[1],
        hidden_layers=config["hidden_layers"],
        dropout=config["dropout"],
    )
    model, history = run_experiment(model, numeric_loaders, config)

    # Посчитайте метрики на test
    test_metrics, _, _ = evaluate(..., ..., ..., ...)

    mlp_rows.append(
        {
            "name": config["name"],
            "best_val_pr_auc": history["pr_auc"].max(),
            **test_metrics,
        }
    )
    mlp_histories[config["name"]] = history
    mlp_models[config["name"]] = model

mlp_results = (
    pd.DataFrame(mlp_rows)
    .sort_values("best_val_pr_auc", ascending=False)
    .reset_index(drop=True)
)

display(mlp_results.round(4))

**Дополнительное задание**

Четвёртую конфигурацию `extra` можно делать после того, как у вас запускаются три базовые конфигурации и появляется таблица результатов. Сдать проект можно и без неё.

Если выполняете допзадание, заполните блок `extra_config` в следующей ячейке. Перед запуском кратко обоснуйте параметры: размер слоёв, `dropout`, `lr` и `weight_decay`.

In [ ]:
# Дополнительное задание
# Когда три базовые конфигурации работают, заполните блок ниже
extra_config = {
  "name": "extra",
  "hidden_layers": ...,  # например, (32, 16)
  "dropout": ...,
  "lr": ...,
  "weight_decay": ...,
  "epochs": 8,
}

model = MLP(
  input_dim=X_train.shape[1],
  hidden_layers=extra_config["hidden_layers"],
  dropout=extra_config["dropout"],
)
model, history = run_experiment(model, numeric_loaders, extra_config)

# Посчитайте метрики на test
test_metrics, _, _ = evaluate(..., ..., ..., ...)

# Добавляем результаты в mlp_rows и пересобираем отчёт
mlp_rows.append(
        {
            "name": extra_config["name"],
            "best_val_pr_auc": history["pr_auc"].max(),
            **test_metrics,
        }
    )

mlp_histories[extra_config["name"]] = history
mlp_models[extra_config["name"]] = model

mlp_results = (
    pd.DataFrame(mlp_rows)
    .sort_values("best_val_pr_auc", ascending=False)
    .reset_index(drop=True)
)

display(mlp_results.round(4))


Если вы выполняли дополнительную конфигурацию `extra`, сопоставьте ожидания с результатом:

* стала ли модель стабильнее;
* улучшился ли PR-AUC;
* как изменились ROC-AUC, precision и recall;
* есть ли признаки переобучения согласно данным кривых loss.

In [ ]:
best_mlp_name = mlp_results.iloc[0]["name"]
best_mlp_model = mlp_models[best_mlp_name]

best_mlp_path = MODELS_DIR / "best_mlp_numeric.pt"

# Сохраните веса лучшей MLP через state_dict
torch.save(..., best_mlp_path)

print(f"Лучшая MLP-конфигурация по PR-AUC на валидационной выборке: {best_mlp_name}")
print(f"state_dict сохранён в: {best_mlp_path}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, history in mlp_histories.items():
    # Постройте train loss и validation loss для каждой конфигурации
    axes[0].plot(..., ..., label=name)
    axes[1].plot(..., ..., label=name)

axes[0].set_title("MLP: train loss")
axes[1].set_title("MLP: validation loss")

for ax in axes:
    ax.set_xlabel("Эпоха")
    ax.set_ylabel("Loss")
    ax.grid(True)
    ax.legend()

plt.tight_layout()
plt.show()

**Вывод по эксперименту 1**

Соберите результаты в одну сравнительную таблицу. Постройте кривые обучения и сравните поведение моделей. Ответьте на вопросы:

* Какая MLP-конфигурация сработала лучше с точки зрения метрики PR-AUC? Что показывают остальные метрики этого запуска?
* Продолжает ли уменьшаться loss на обучающей выборке, когда loss на валидационной выборке уже растёт или перестаёт улучшаться?
* Помогли ли слои `Dropout` и `weight_decay` уменьшить переобучение?
* Почему числовых признаков может быть недостаточно для анализа тональности?

Если вы выполняли дополнительное задание, также расскажите, совпали ли ожидания от конфигурации с результатом.

## Шаг 5. Эксперимент 2 — Embedding + mean pooling + MLP

В ходе второго эксперимента вы передадите в нейросеть тексты отзывов, а не только их числовые признаки. Используйте те же обучающую, валидационную и тестовую выборки, что и в первом эксперименте.

Далее вам нужно:

1. Подготовить текст — использовать готовую токенизацию, построить словарь только по обучающей выборке и перевести слова в индексы.
2. Собрать `Dataset` для текстов — реализовать длину датасета, получение одного объекта, обрезание длинных отзывов и пэддинг коротких отзывов до `MAX_LENGTH`.
3. Описать модель — получить эмбеддинги, исключить PAD-токены маской, усреднить эмбеддинги настоящих токенов и передать итоговый вектор в MLP.
4. Обучить модель через уже готовый `run_experiment`, оценить качество на тестовой выборке и сохранить веса через `state_dict`. Новый цикл обучения писать не нужно.
5. Провести анализ ошибок: найти конкретные примеры ответов типа FP и FN, чтобы объяснить, почему модель могла ошибиться.

### 5.1. Готовая подготовка текста

Код для токенизации и построения словаря уже почти полностью готов. Ваша задача — запустить ячейку и проверить, что словарь действительно строится **только по обучающей выборке**.

Обратите внимание на три специальных элемента логики:

- `PAD_IDX` — индекс для добивки коротких текстов до одной длины.
- `UNK_IDX` — индекс для редких или неизвестных слов. Подробнее этот элемент вы рассмотрите в следующем спринте.
- `text_to_indices` — функция, которая превращает текст в список индексов словаря.

После запуска проверьте размер словаря и первые элементы: это быстрый способ убедиться, что подготовка текста прошла корректно.

In [ ]:
TOKEN_RE = re.compile(r"[а-яёa-z0-9]+", flags=re.IGNORECASE)
PAD_IDX = 0
UNK_IDX = 1


def tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall(str(text).lower())


def build_vocab(texts, min_freq: int = 2, max_size: int = 20_000) -> dict:
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))

    vocab = {"<PAD>": PAD_IDX, "<UNK>": UNK_IDX}
    for token, count in counter.most_common(max_size - len(vocab)):
        if count >= min_freq:
            vocab[token] = len(vocab)

    return vocab


def text_to_indices(text: str, vocab: dict) -> list[int]:
    return [vocab.get(token, UNK_IDX) for token in tokenize(text)]


vocab = build_vocab(train_df["text"], min_freq=2, max_size=20_000)
print(f"Размер словаря: {len(vocab)}")
print("Первые элементы словаря:", list(vocab.items())[:10])

### 5.2. Dataset и пэддинг

В этой части нужно дописать класс `ReviewsTextDataset`:

1. Верните количество текстов в датасете в `__len__`.
2. Возьмите текст по индексу и переведите его в индексы токенов в `__getitem__`.
3. Обрежьте слишком длинные отзывы до `MAX_LENGTH`.
4. Дополните короткие отзывы значением `PAD_IDX`, чтобы все последовательности в батче были одинаковой длины.
5. Создайте `DataLoader` для обучающей, валидационной и тестовой выборок.
6. Проверьте форму первого батча: ожидается матрица размера `batch_size` × `MAX_LENGTH` и тип `torch.long` для индексов токенов.

In [ ]:
class ReviewsTextDataset(Dataset):
    def __init__(self, texts, target, vocab: dict, max_length: int = 120):
        self.texts = list(texts)
        self.target = np.array(target, dtype=np.float32)
        self.vocab = vocab
        self.max_length = max_length

    def __len__(self):
        # Верните количество текстов
        return ...

    def __getitem__(self, idx):
        ids = text_to_indices(self.texts[idx], self.vocab)

        # Обрежьте последовательность до max_length
        ids = ...

        # Если последовательность короче max_length, добавьте PAD_IDX
        if ...:
            ids = ...

        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(self.target[idx], dtype=torch.float32),
        )


MAX_LENGTH = 120

text_loaders = {
    "train": DataLoader(
        ReviewsTextDataset(...),
        batch_size=BATCH_SIZE,
        shuffle=True,
    ),
    "val": DataLoader(
        ReviewsTextDataset(...),
        batch_size=BATCH_SIZE,
        shuffle=False,
    ),
    "test": DataLoader(
        ReviewsTextDataset(...),
        batch_size=BATCH_SIZE,
        shuffle=False,
    ),
}

ids_batch, text_target_batch = next(iter(text_loaders["train"]))
print("ids_batch:", ids_batch.shape, ids_batch.dtype)
print("target_batch:", text_target_batch.shape, text_target_batch.dtype)

### 5.3. Модель с Embedding и mean pooling

В этой части нужно дописать `forward` модели `EmbeddingMLP`.

Что вам нужно сделать внутри `forward`:

1. Передать `input_ids` в `nn.Embedding`, чтобы каждому индексу токена соответствовал обучаемый вектор.
2. Умножить эмбеддинги на маску и посчитать длину каждого текста без PAD-токенов.
3. Выполнить mean pooling: усреднить эмбеддинги настоящих токенов и получить один вектор на отзыв.
4. Передать этот вектор в MLP-классификатор и вернуть логит.

In [ ]:
class EmbeddingMLP(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int = 64,
        hidden_dim: int = 64,
        dropout: float = 0.3,
        pad_idx: int = PAD_IDX,
    ):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_idx,
        )
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )
        self.pad_idx = pad_idx

    def forward(self, input_ids):
        # 1. Получите эмбеддинги
        embeddings = ...

        # 2. PAD-токены нужны только для выравнивания длин
        # Маска помогает не учитывать их при усреднении
        mask = (input_ids != self.pad_idx).unsqueeze(-1)
        masked_embeddings = embeddings * mask
        lengths = mask.sum(dim=1).clamp(min=1)

        # 3. Выполните mean pooling и передайте результат в MLP
        pooled = ...
        logits = ...
        return ...

### 5.4. Обучение, оценка и сохранение

Используйте тот же `run_experiment`, что и для MLP на числовых признаках. Это важно: цикл обучения и цикл валидации не нужно писать второй раз.

Что нужно сделать в коде:

1. Проверьте конфигурацию текстовой модели: размер эмбеддинга, размер скрытого слоя, `dropout`, `lr` (learning rate), `weight_decay` и число эпох.
2. Создайте объект `EmbeddingMLP` с нужным размером словаря.
3. Запустите обучение через `run_experiment`.
4. Посчитайте метрики на тестовой выборке с помощью `evaluate`.
5. Постройте график loss на обучающей и валидационной выборках.
6. Сохраните веса обученной текстовой модели через `state_dict`.

In [ ]:
text_config = {
    "embedding_dim": 64,
    "hidden_dim": 64,
    "dropout": 0.3,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "epochs": 8,
}

set_seed(RANDOM_STATE)

text_model = EmbeddingMLP(
    vocab_size=len(vocab),
    embedding_dim=text_config["embedding_dim"],
    hidden_dim=text_config["hidden_dim"],
    dropout=text_config["dropout"],
)

text_model, text_history = run_experiment(text_model, text_loaders, text_config)
text_test_metrics, y_true_text_test, y_prob_text_test = evaluate(
    text_model,
    text_loaders["test"],
    criterion,
    device,
)

display(pd.DataFrame([text_test_metrics]).round(4))

In [ ]:
plt.figure(figsize=(7, 4))

# Постройте train loss и validation loss для текстовой модели
plt.plot(..., ..., label="train loss")
plt.plot(..., ..., label="validation loss")

plt.title("Embedding + mean pooling + MLP")
plt.xlabel("Эпоха")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
best_text_path = MODELS_DIR / "best_embedding_mlp.pt"

# Сохраните веса текстовой модели через state_dict
torch.save(..., best_text_path)

print(f"state_dict текстовой модели сохранён в: {best_text_path}")

### 5.5. Анализ ошибок текстовой модели

После обучения нужно посмотреть на конкретные ошибки, чтобы разобраться, что именно не ухватывает модель.

Что нужно найти:

- Ложноположительные примеры (FP) — модель посчитала отзыв негативным (класс 1), хотя его истинный класс равен 0.
- Ложноотрицательные примеры (FN) — модель не распознала негативный отзыв (класс 1).

Для каждого типа ошибки проанализируйте текст, вероятность негативного отзыва и возможную причину. Обратите внимание на смешанную тональность, короткие отзывы, отрицания, сарказм, длинный контекст и потерю порядка слов при mean pooling.

Также учитывайте возможный шум разметки: `target` получен из пользовательской оценки, а не из ручного анализа текста.

In [ ]:
error_analysis = test_df[["text", "target", "review_len_words"]].copy()
error_analysis["prob_negative"] = y_prob_text_test
error_analysis["prediction"] = (error_analysis["prob_negative"] >= 0.5).astype(int)

# Найдите false positive: модель считает отзыв негативным, но target = 0
# Полезно отсортировать такие примеры по prob_negative по убыванию
false_positive = ...

# Найдите false negative: модель пропускает негативный отзыв
# Полезно отсортировать такие примеры по prob_negative по возрастанию
false_negative = ...

print("False positive: модель считает отзыв негативным, но target = 0")
display(false_positive)

print("False negative: модель пропускает негативный отзыв")
display(false_negative)

**Анализ найденных ошибок**

Заполните два коротких списка:

- FP — какие слова или фрагменты могли подтолкнуть модель посчитать отзыв отрицательным, хотя `target = 0`.
- FN — почему модель могла пропустить отзыв класса 1. Возможные причины: редкие слова, длинный контекст, отрицания или слабые негативные маркеры.

Отдельно отметьте, могли ли дисбаланс классов и разметка по пользовательским оценкам повлиять на ошибки: отзывы с оценкой 3–5 попадают в `target = 0`, хотя текст иногда бывает нейтральным или смешанным.

**Вывод по эксперименту 2**

Ответьте на вопросы:

* Насколько текстовая модель лучше или хуже MLP на числовых признаках по PR-AUC? Что показывают остальные метрики?
* Какие ошибки вы нашли среди false positive и false negative?
* Какие причины ошибок кажутся наиболее вероятными?
* Какие ограничения остаются у mean pooling?
* Почему текстовое представление обычно даёт модели больше информации, чем поверхностные числовые признаки?

## Шаг 6. Итоговое сравнение

Для честного выбора лучшей числовой MLP используйте PR-AUC на валидационной выборке, потому что положительный класс редкий. ROC-AUC, precision, recall, F1 и accuracy нужны для дополнительной диагностики. Метрики на тестовой выборке анализируем уже после выбора модели.

Ниже соберите сравнительную таблицу минимум с двумя строками: лучшая MLP на числовых признаках и `Embedding + pooling + MLP`.

In [ ]:
final_comparison = pd.DataFrame(
    [
        {
            "Подход": "MLP на числовых признаках",
            "PR-AUC test": ...,
            "ROC-AUC test": ...,
            "Precision": ...,
            "Recall": ...,
            "F1": ...,
            "Ограничения": "...",
        },
        {
            "Подход": "Embedding + pooling + MLP",
            "PR-AUC test": ...,
            "ROC-AUC test": ...,
            "Precision": ...,
            "Recall": ...,
            "F1": ...,
            "Ограничения": "...",
        },
    ]
)

display(final_comparison)

## Итоговые выводы

Сформулируйте итоговые выводы. Ответьте на вопросы:

* Какой подход показал лучшее качество?
* По каким метрикам преимущество особенно заметно?
* Что текстовая модель узнала из того, что упускала модель на числовых признаках?
* В каких ситуациях числовые признаки всё же могут быть полезны?
* Какие ограничения есть у mean pooling?
* Как на результат могли повлиять дисбаланс классов и разметка, полученная из пользовательских оценок?
* Какой следующий эксперимент вы бы предложили?

## Чек-лист самопроверки

До сдачи проекта ревьюеру убедитесь в том, что выполнили все указанные ниже действия:

- Разделение на обучающую, валидационную и тестовую выборки выполнено со стратификацией в пропорции 70:15:15.
- Оба эксперимента используют одно и то же разделение на выборки.
- Числовые признаки из текста извлечены через pandas.
- `federal_district` преобразован с помощью One-Hot Encoder.
- `StandardScaler` и `OneHotEncoder` обучены только на обучающей выборке.
- `Dataset` и `DataLoader` реализованы для числового и текстового эксперимента.
- Цикл обучения содержит порядок `zero_grad` → `forward` → `loss` → `backward` → `step`.
- Цикл валидации использует `model.eval()` и `torch.no_grad()`.
- Проведено минимум три конфигурации MLP.
- При желании выполнена дополнительная четвёртая конфигурация MLP. Она сопровождается обоснованием выбора.
- Есть графики loss на обучающей и валидационной выборках.
- Есть метрики на валидационной и тестовой выборках.
- Лучшая модель сохранена через `state_dict`.
- Есть сравнительная таблица метрик на тестовой выборке.
- Есть анализ ошибок текстовой модели с конкретными примерами и объяснением возможных причин.
- Написаны выводы по EDA, обоим экспериментам и итоговому сравнению.